# Worked examples

Six real ledger lines, computed step by step from `data/`, without the pipeline. The numbers are the ones in
`## Worked examples` of [`docs/design.md`](../docs/design.md). Each example, W01 to W06, ends with **Result:**.
The last section compares every row with the pipeline's `output/allocation.csv` (F5).

- This notebook uses pandas; the pipeline uses SQL. Two separate computations that give the same rows are the check.
- IDs (R01 rules, Q01 open questions, I01 data issues, ...) are explained in [`docs/design.md`](../docs/design.md#ids).

## Setup

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data")

# Read as text, like the exploration notebook
ledger = pd.read_csv(DATA / "ledger.csv", dtype=str)
entities = pd.read_csv(DATA / "entities.csv", dtype=str)
accounts = pd.read_csv(DATA / "accounts.csv", dtype=str)
cost_centers = pd.read_csv(DATA / "cost_centers.csv", dtype=str)
pools = pd.read_csv(DATA / "pools.csv", dtype=str)
headcount_games = pd.read_csv(DATA / "headcount_games.csv", dtype=str)
headcount_employees = pd.read_csv(DATA / "headcount_employees.csv", dtype=str)
game_mapping = pd.read_csv(DATA / "game_mapping.csv", dtype=str)

# The only cleaning: trim the one untrimmed name found in 01-explore, C02 (I02).
# The pipeline trims every name column used to match tables (Q08).
game_mapping["game_group"] = game_mapping["game_group"].str.strip()

ledger["amount_usd"] = ledger["amount_usd"].astype(float)
headcount_games["fte"] = headcount_games["fte"].astype(float)

In [2]:
# One ledger line with its entity group, account and cost center
def line(line_id):
    row = ledger[ledger["line_id"] == line_id]
    row = row.merge(entities, on="entity").merge(accounts, on="account").merge(cost_centers, on="cost_center", how="left")
    return row.iloc[0]


# The mapping row of a game or bucket that applies to a month (R10)
def mapping(name, month):
    first_day = month + "-01"
    rows = game_mapping[
        (game_mapping["game_group"] == name)
        & (game_mapping["start_date"] <= first_day)
        & (game_mapping["end_date"] >= first_day)
    ]
    assert len(rows) <= 1, f"two mapping rows for {name} in {month} (Q03)"
    return rows


# Every example keeps its rows here; the last section compares them with the pipeline
expected = []


def keep(rows):
    expected.append(rows[["line_id", "game_or_bucket", "amount", "bearing_group", "side"]])

### W01. Direct, single game (R04, R06, R07)

In [3]:
l = line("L00105")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["amount_usd"])

# The cost center has a game, so the whole line goes to it (share 1)
m = mapping(l["game"], l["month"])
result = pd.DataFrame([{
    "game": l["game"], "share": 1, "amount": round(l["amount_usd"], 2),
    "bearing_group": l["entity_group"],  # direct: bearing group = booked group (R06)
    "side": m["side"].iloc[0],
}])
keep(result.rename(columns={"game": "game_or_bucket"}).assign(line_id="L00105"))
result

2026-01 LGB Salaries 1001 Star Racer 103255.1


,game,share,amount,bearing_group,side
0,Star Racer,1,103255.1,BR,BR Games


**Result:** Star Racer gets 103,255.10 USD, bearing group BR, side BR Games. The mapping row only matches after the trim
(I02).

### W02. Direct, pool (R05)

In [4]:
l = line("L00054")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["pool_id"], l["amount_usd"])

# The pool's games, and their FTE that month
pool_games = pools[pools["pool_id"] == l["pool_id"]]
fte = pool_games.merge(headcount_games[headcount_games["month"] == l["month"]], on="game", how="left")
print(fte[["game", "fte"]])

# Only games with FTE > 0 share the line
fte = fte[fte["fte"] > 0].copy()
total_fte = fte["fte"].sum()
fte["share"] = fte["fte"].map("{:g}".format) + " / " + f"{total_fte:g}"
fte["amount"] = (l["amount_usd"] * fte["fte"] / total_fte).round(2)
fte["bearing_group"] = l["entity_group"]
fte["side"] = [mapping(g, l["month"])["side"].iloc[0] for g in fte["game"]]
fte = fte.sort_values("game")
print("total:", round(fte["amount"].sum(), 2))
keep(fte.rename(columns={"game": "game_or_bucket"}).assign(line_id="L00054"))
fte[["game", "share", "amount", "bearing_group", "side"]]

2026-01 LGB Salaries 2010 New Games Prototyping NEW_SHARED 21972.14
          game   fte
0  Puzzle Pals  11.5
1   Moon Miner   NaN
2   Pixel Farm  14.5
3   Rune Quest   9.0
4    Tiny Chef   6.0
total: 21972.14


,game,share,amount,bearing_group,side
2,Pixel Farm,14.5 / 41,7770.63,BR,US Games
0,Puzzle Pals,11.5 / 41,6162.92,BR,BR Games
3,Rune Quest,9 / 41,4823.15,BR,US Games
4,Tiny Chef,6 / 41,3215.44,BR,BR Games


**Result:**
- Moon Miner has no FTE in January, so it gets nothing; the other four share 41 FTE.
- Puzzle Pals and Tiny Chef go to BR Games, Pixel Farm and Rune Quest to US Games. All four keep bearing group BR.
- The rounded pieces add up to the line, 21,972.14 USD.

### W03. Indirect (R08, R09, Q04)

In [5]:
l = line("L00444")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["amount_usd"])

# Bucket of each employee that month: first word of the category + entity group, in lower case
employees = headcount_employees[headcount_employees["month"] == l["month"]].merge(entities, on="entity", how="left")
employees["entity_group"] = employees["entity_group"].fillna("unknown")  # D10: live_unknown, new_unknown
employees["bucket"] = employees["category"].str.split().str[0].str.lower() + "_" + employees["entity_group"].str.lower()

buckets = employees.groupby("bucket").size().reset_index(name="employees")
total = buckets["employees"].sum()
buckets["share"] = buckets["employees"].astype(str) + " / " + str(total)
buckets["amount"] = (l["amount_usd"] * buckets["employees"] / total).round(2)
buckets["bearing_group"] = [mapping(b, l["month"])["bearing_group"].iloc[0] for b in buckets["bucket"]]
buckets["side"] = [mapping(b, l["month"])["side"].iloc[0] for b in buckets["bucket"]]
print("total:", round(buckets["amount"].sum(), 2))
keep(buckets.rename(columns={"bucket": "game_or_bucket"}).assign(line_id="L00444"))
buckets[["bucket", "share", "amount", "bearing_group", "side"]]

2026-02 LGB Salaries 3301 Finance 21346.02
total: 21346.02


,bucket,share,amount,bearing_group,side
0,live_br,71 / 123,12321.69,BR,BR Games
1,new_br,25 / 123,4338.62,BR,BR Games
2,new_fi,8 / 123,1388.36,FI,US Games
3,new_others,5 / 123,867.72,Others,US Games
4,new_us,14 / 123,2429.63,US,US Games


**Result:** 123 employees in February, in five buckets. The bearing group and side come from each bucket's mapping row.
The rounded pieces add up to the line, 21,346.02 USD.

### W04. By entity (R07)

In [6]:
rows = []
for line_id in ["L00334", "L00336"]:
    l = line(line_id)
    side = mapping(l["game"], l["month"])["side"].iloc[0]
    print(line_id, l["game"], "mapping side:", side)
    # By entity: BR Games when booked by LGB, US Games for any other entity
    if side == "By entity":
        side = "BR Games" if l["entity"] == "LGB" else "US Games"
    rows.append({"line": line_id, "month": l["month"], "entity": l["entity"], "account": l["account_name"],
                 "cost_center": l["cost_center"], "amount": l["amount_usd"],
                 "bearing_group": l["entity_group"], "side": side})
result = pd.DataFrame(rows)
keep(result.rename(columns={"line": "line_id"}).assign(game_or_bucket=l["game"]))
result

L00334 Legacy Arcade mapping side: By entity
L00336 Legacy Arcade mapping side: By entity


,line,month,entity,account,cost_center,amount,bearing_group,side
0,L00334,2026-01,LGB,Salaries,1004,24203.53,BR,BR Games
1,L00336,2026-01,LGI,Salaries,1004,12825.36,US,US Games


**Result:** Same game and month: booked by LGB it goes to BR Games, booked by LGI to US Games.

### W05. Mapping dates (R10)

In [7]:
rows = []
for line_id in ["L00114", "L00494"]:
    l = line(line_id)
    side = mapping(l["game"], l["month"])["side"].iloc[0]
    rows.append({"line": line_id, "month": l["month"], "entity": l["entity"], "account": l["account_name"],
                 "cost_center": l["cost_center"], "amount": l["amount_usd"],
                 "bearing_group": l["entity_group"], "side": side})
print(game_mapping[game_mapping["game_group"] == "Puzzle Pals"][["game_group", "side", "start_date", "end_date"]])
result = pd.DataFrame(rows)
keep(result.rename(columns={"line": "line_id"}).assign(game_or_bucket=l["game"]))
result

    game_group      side  start_date    end_date
4  Puzzle Pals  BR Games  2025-01-01  2026-02-28
5  Puzzle Pals  US Games  2026-03-01  9999-12-31


,line,month,entity,account,cost_center,amount,bearing_group,side
0,L00114,2026-02,LGB,Salaries,2001,31929.39,BR,BR Games
1,L00494,2026-03,LGB,Salaries,2001,32256.22,BR,US Games


**Result:** Puzzle Pals is BR Games in February and US Games in March. The bearing group stays BR: only the side changes.

### W06. Unallocated (Q01, I01, I03)

In [8]:
# Whole line: the cost center is not in cost_centers.csv
l = line("L00169")
print(l["line_id"], l["month"], l["entity"], l["account_name"], l["cost_center"], l["amount_usd"])
print("department:", l["department"])
keep(pd.DataFrame([{"line_id": "L00169", "game_or_bucket": None, "amount": l["amount_usd"],
                    "bearing_group": None, "side": None}]))

L00169 2026-02 LGB Professional Services 3999 17017.98
department: nan


**Result:** No department, so the line has no path: one row, share 1, `unallocated`, `unknown_cost_center` (I01).

In [9]:
# One piece: a bucket with no mapping row
l = line("L00234")
print(l["month"], l["entity"], l["account_name"], l["cost_center"], l["cost_center_name"], l["amount_usd"])

employees = headcount_employees[headcount_employees["month"] == l["month"]].merge(entities, on="entity", how="left")
employees["entity_group"] = employees["entity_group"].fillna("unknown")  # D10: live_unknown, new_unknown
employees["bucket"] = employees["category"].str.split().str[0].str.lower() + "_" + employees["entity_group"].str.lower()

buckets = employees.groupby("bucket").size().reset_index(name="employees")
total = buckets["employees"].sum()
buckets["share"] = buckets["employees"].astype(str) + " / " + str(total)
buckets["amount"] = (l["amount_usd"] * buckets["employees"] / total).round(2)
buckets["has_mapping"] = [len(mapping(b, l["month"])) == 1 for b in buckets["bucket"]]
buckets["status"] = buckets["has_mapping"].map({True: "allocated", False: "unallocated"})
buckets["reason"] = buckets["has_mapping"].map({True: "", False: "no_mapping_row"})
# Bearing group and side only on allocated rows (Output, allocation.csv)
buckets["bearing_group"] = [mapping(b, l["month"])["bearing_group"].iloc[0] if ok else None
                            for b, ok in zip(buckets["bucket"], buckets["has_mapping"])]
buckets["side"] = [mapping(b, l["month"])["side"].iloc[0] if ok else None
                   for b, ok in zip(buckets["bucket"], buckets["has_mapping"])]
print("total:", round(buckets["amount"].sum(), 2))
keep(buckets.rename(columns={"bucket": "game_or_bucket"}).assign(line_id="L00234"))
buckets[["bucket", "share", "amount", "bearing_group", "side", "status", "reason"]]

2026-03 LGB Salaries 3301 Finance 24080.73
total: 24080.73


,bucket,share,amount,bearing_group,side,status,reason
0,live_br,70 / 125,13485.21,BR,BR Games,allocated,
1,live_fi,2 / 125,385.29,NaN,NaN,unallocated,no_mapping_row
2,new_br,26 / 125,5008.79,BR,BR Games,allocated,
3,new_fi,8 / 125,1541.17,FI,US Games,allocated,
4,new_others,5 / 125,963.23,Others,US Games,allocated,
5,new_us,14 / 125,2697.04,US,US Games,allocated,


**Result:** live_fi (2 of 125 employees) has no mapping row, so its piece of 385.29 USD is `unallocated`, `no_mapping_row`
(I03). The other five pieces are allocated, and all six add up to the line, 24,080.73 USD.

## Compare with the pipeline (F5)

Run `python run.py` from the repo root first, so `output/` is current.

In [10]:
# The notebook's rows, and the pipeline's rows for the same ledger lines
mine = pd.concat(expected).fillna("")
mine["amount"] = mine["amount"].astype(float).round(2)

allocation = pd.read_csv("../output/allocation.csv", dtype=str).fillna("")
pipeline = allocation[allocation["line_id"].isin(mine["line_id"])]
pipeline = pipeline[["line_id", "game_or_bucket", "amount_usd", "bearing_group", "side"]].copy()
pipeline["amount_usd"] = pipeline["amount_usd"].astype(float)

# One row per piece; a piece only one side has shows up as left_only or right_only
both = mine.merge(pipeline, on=["line_id", "game_or_bucket"], how="outer", suffixes=("", "_pipeline"), indicator=True)
both["match"] = (
    (both["_merge"] == "both")
    & (both["amount"] == both["amount_usd"])
    & (both["bearing_group"] == both["bearing_group_pipeline"])
    & (both["side"] == both["side_pipeline"])
)
print("rows:", len(both), "| match:", both["match"].sum(), "| differ:", (~both["match"]).sum())
both[["line_id", "game_or_bucket", "amount", "amount_usd", "bearing_group", "bearing_group_pipeline",
      "side", "side_pipeline", "match"]]

rows: 21 | match: 21 | differ: 0


,line_id,game_or_bucket,amount,amount_usd,bearing_group,bearing_group_pipeline,side,side_pipeline,match
0,L00054,Pixel Farm,7770.63,7770.63,BR,BR,US Games,US Games,True
1,L00054,Puzzle Pals,6162.92,6162.92,BR,BR,BR Games,BR Games,True
2,L00054,Rune Quest,4823.15,4823.15,BR,BR,US Games,US Games,True
3,L00054,Tiny Chef,3215.44,3215.44,BR,BR,BR Games,BR Games,True
4,L00105,Star Racer,103255.10,103255.10,BR,BR,BR Games,BR Games,True
5,L00114,Puzzle Pals,31929.39,31929.39,BR,BR,BR Games,BR Games,True
6,L00169,,17017.98,17017.98,,,,,True
7,L00234,live_br,13485.21,13485.21,BR,BR,BR Games,BR Games,True
8,L00234,live_fi,385.29,385.29,,,,,True
9,L00234,new_br,5008.79,5008.79,BR,BR,BR Games,BR Games,True


**Result:** All 21 rows of the six examples match the pipeline: same pieces, amounts to the cent, bearing group and side.
The unallocated pieces (L00169 and live_fi) have no bearing group or side in both.